# Fitting the pair source with a normalizing flow (PyTorch version)

PyTorch port of `flow-optimizer.ipynb` (TensorFlow + TensorFlow Probability). No TFP is needed: all flow layers are written here
(they have a generative direction `forward` and a density direction `inverse`, each returning the log-determinant), so the notebook needs only
`torch`, `numpy` and `matplotlib`.

Fits a model of the 3D pair-separation source `S(r)` so that `C(q) = < K(|q|, |r|, angle(q, r)) >_{r ~ S}` reproduces the tabulated correlation function
`65.dat`. `K` is the kernel `Kl-ppiplus.dat` (100x100x100 table in `|q|`, `|r|`, `theta`, trilinearly interpolated, constant outside the grid).
The fitted sources are compared with the *true* source samples `ppi-0.65.dat` (500k points `(x, y, z)` in fm).

Models (all start exactly at the Gaussian initial guess):
1. **Gaussian** - `r = L z + mu`.
2. **RealNVP** - affine coupling layers (bounded log-scale) followed by the same affine map.
3. **Levy** - per-axis alpha-stable latent with **trained** stability index `alpha` and skewness `beta`, then the affine map.
4. **sas** - affine / sinh-arcsinh layers (trainable skewness and tail weight).
5. **nsf** - rational-quadratic **spline** couplings (neural spline flow).

Then the same models are fitted **directly to the density** of the true source (no correlator): maximum likelihood for the Gaussian-base flows,
sliced Wasserstein-1 for the Levy model (no closed-form density). The last cell compares everything.

Install: `pip install torch numpy matplotlib jupyter` (no TensorFlow / TFP / Keras). Everything runs on the CPU; the models are tiny.
All the maths in the layers was tested against NumPy references (kernel interpolation, spline forward / inverse / log-det, sinh-arcsinh, Levy sampler);
the first cells below re-run these checks with autograd on your machine.

In [ ]:
import os, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.rcParams.update({'xtick.top': True, 'ytick.right': True, 'xtick.direction': 'in', 'ytick.direction': 'in',
                     'xtick.minor.visible': True, 'ytick.minor.visible': True})

DTYPE = torch.float32
DEVICE = torch.device('cpu')          # the models are tiny; the cost is the kernel gather, which is fine on the CPU
torch.set_num_threads(os.cpu_count() or 4)
torch.manual_seed(0); np.random.seed(0)

def T(x):
    """numpy / list -> float tensor on DEVICE"""
    return torch.as_tensor(np.asarray(x), dtype=DTYPE, device=DEVICE)

print('torch', torch.__version__, '| device', DEVICE, '| threads', torch.get_num_threads())

## Kernel `K(|q|, |r|, theta)`: trilinear interpolation (differentiable)

In [ ]:
class KernelInterp:
    """Trilinear interpolation of a regular (|q|, |r|, theta) table with constant extension outside the grid
    (what tfp.math.batch_interp_regular_nd_grid did). Differentiable w.r.t. the coordinates."""
    def __init__(self, table, xmax=(100.0, 40.0, math.pi)):
        self.n = table.shape[0]
        self.table = T(table).reshape(-1)
        self.scale = T([(self.n - 1) / x for x in xmax])               # coordinate -> grid index
    def __call__(self, coords):                                         # coords [..., 3]
        n = self.n
        u = torch.clamp(coords * self.scale, 0.0, float(n - 1))
        i0 = torch.clamp(torch.floor(u).long(), max=n - 2)              # lower corner
        f = u - i0.to(u.dtype)                                          # fractional position, carries the gradient
        out = 0.0
        for da in (0, 1):
            wa = f[..., 0] if da else 1 - f[..., 0]
            for db in (0, 1):
                wb = f[..., 1] if db else 1 - f[..., 1]
                for dc in (0, 1):
                    wc = f[..., 2] if dc else 1 - f[..., 2]
                    idx = ((i0[..., 0] + da) * n + (i0[..., 1] + db)) * n + (i0[..., 2] + dc)
                    out = out + wa * wb * wc * self.table[idx]
        return out

y_values = np.reshape(np.loadtxt("Kl-ppiplus.dat"), (100, 100, 100))      # axes: |q|, |r|, theta
interp = KernelInterp(y_values)                                          # |q| [MeV/c], |r| [fm], theta [rad]

# self-test: a linear table must be reproduced exactly inside the grid, and the gradient must be the slope
lin = 0.5 * np.arange(100)[:, None, None] - 0.25 * np.arange(100)[None, :, None] + 0.1 * np.arange(100)[None, None, :] + 3
lin_interp = KernelInterp(lin)
pts = ((torch.rand(2000, 3, dtype=DTYPE) * 0.98 + 0.01) * T([100.0, 40.0, math.pi])).requires_grad_(True)
val = lin_interp(pts)
expect = (pts * lin_interp.scale) @ T([0.5, -0.25, 0.1]) + 3
grad, = torch.autograd.grad(val.sum(), pts)
print('interpolation error', float((val - expect).abs().max()),
      '| gradient error', float((grad - lin_interp.scale * T([0.5, -0.25, 0.1])).abs().max()))

## Data: q grid, target correlation function, true source samples

In [ ]:
q_max, Nqs = 50, 20
g = (-q_max + 2 * q_max / Nqs * np.arange(Nqs)).astype(np.float32)
qs_np = np.stack(np.meshgrid(g, g, g, indexing='ij'), axis=-1).reshape(-1, 3)
true_Cq = np.loadtxt("65.dat", dtype=np.float32)
nonzero = np.where(np.linalg.norm(qs_np, axis=1) > 0)[0]                  # q = 0 has no direction
train_idx = nonzero[::8]                                                   # 1000 q points

true_source = np.loadtxt("ppi-0.65.dat")                                   # 500k samples of the real source [fm]
print('true source: mean', true_source.mean(0).round(2), ' std', true_source.std(0).round(2))

qs_tr, c_tr = T(qs_np[train_idx]), T(true_Cq[train_idx])

init_mean = np.array([4.0, 0.0, -0.5], dtype=np.float32)
init_cov = np.diag([80.0, 30.0, 30.0]).astype(np.float32)

## Loss: `|| <K>_r - C ||`  (numerically safe)

`acos` has an infinite slope at +-1, so the ratio is clipped inside `[-1+1e-6, 1-1e-6]`; magnitudes get a tiny epsilon; `q = 0` is excluded from the grid.

In [ ]:
def compute_loss(xs, true_ys, qs):
    eps = 1e-6
    r = xs.norm(dim=1) + eps                                               # [N]
    q = qs.norm(dim=1) + eps                                               # [M]
    ratio = (qs @ xs.T) / (q[:, None] * r[None, :])                        # [M, N]
    angle = torch.acos(torch.clamp(ratio, -1.0 + 1e-6, 1.0 - 1e-6))
    coords = torch.stack([q[:, None].expand_as(angle), r[None, :].expand_as(angle), angle], dim=-1)   # [M, N, 3]
    return torch.linalg.norm(interp(coords).mean(dim=1) - true_ys)


@torch.no_grad()
def correlation_np(src, q_points, chunk=100):
    """C(q) of a set of source samples (numpy in, numpy out)"""
    xs, out = T(src), []
    for i in range(0, len(q_points), chunk):
        qc = T(q_points[i:i + chunk])
        r = xs.norm(dim=1) + 1e-6; q = qc.norm(dim=1) + 1e-6
        ang = torch.acos(torch.clamp((qc @ xs.T) / (q[:, None] * r[None, :]), -1 + 1e-6, 1 - 1e-6))
        coords = torch.stack([q[:, None].expand_as(ang), r[None, :].expand_as(ang), ang], dim=-1)
        out.append(interp(coords).mean(dim=1).cpu().numpy())
    return np.concatenate(out)

## Flow layers

Every layer maps latent -> data with `forward(x) -> (y, log|det dy/dx|)` and back with `inverse(y) -> (x, log|det dx/dy|)`.
A `Flow` applies its layers in order to samples of `N(0, I)`; `log_prob(y)` runs the inverses in reverse order (change of variables).

* `Affine`: `y = L x + shift`, `L` lower triangular with positive diagonal (softplus + 1e-3).
* `Permute`: fixed rotation of the coordinates between couplings.
* `AffineCoupling` (RealNVP): `y_b = x_b exp(s(x_a)) + t(x_a)`, `s = 1.5 tanh(.)`; last layer zero-initialised, so it starts as the identity.
* `SplineCoupling` (neural spline flow): monotone rational-quadratic spline on `[-B, B]` per transformed coordinate (identity outside), 16 bins,
  conditioner = small tanh MLP; starts as the identity.
* `SinhArcsinh`: `y = sinh(d asinh(x) + e)` per axis, `d = exp(log_d)`: `d > 1` fattens the tails (stretched-exponential, not power-law), `e` skews.

In [ ]:
class Affine(nn.Module):
    """y = L x + shift;  L lower triangular with strictly positive diagonal."""
    def __init__(self, mean, cov):
        super().__init__()
        L0 = np.linalg.cholesky(np.asarray(cov, dtype=np.float64))
        il, jl = np.tril_indices(3, -1)                                      # (1,0), (2,0), (2,1)
        self.register_buffer('il', torch.as_tensor(il)); self.register_buffer('jl', torch.as_tensor(jl))
        self.raw_diag = nn.Parameter(T(np.log(np.expm1(np.diag(L0) - 1e-3))))      # inverse softplus
        self.off_diag = nn.Parameter(T(L0[il, jl]))
        self.shift = nn.Parameter(T(mean))
    def diag(self):
        return F.softplus(self.raw_diag) + 1e-3
    def L(self):
        return torch.diag(self.diag()) + self.raw_diag.new_zeros(3, 3).index_put((self.il, self.jl), self.off_diag)
    def forward(self, x):
        return x @ self.L().T + self.shift, torch.log(self.diag()).sum().expand(x.shape[0])
    def inverse(self, y):
        Linv = torch.linalg.solve_triangular(self.L(), torch.eye(3, dtype=y.dtype, device=y.device), upper=False)
        return (y - self.shift) @ Linv.T, -torch.log(self.diag()).sum().expand(y.shape[0])


class Permute(nn.Module):
    def __init__(self, perm):
        super().__init__()
        self.register_buffer('perm', torch.as_tensor(perm)); self.register_buffer('inv', torch.as_tensor(np.argsort(perm)))
    def forward(self, x):
        return x[:, self.perm], x.new_zeros(x.shape[0])
    def inverse(self, y):
        return y[:, self.inv], y.new_zeros(y.shape[0])


class SinhArcsinh(nn.Module):
    """y = sinh(d * asinh(x) + e) per axis; d = exp(log_d). d = 1, e = 0 is the identity."""
    def __init__(self):
        super().__init__()
        self.log_d = nn.Parameter(torch.zeros(3)); self.e = nn.Parameter(torch.zeros(3))
    def forward(self, x):
        u = torch.exp(self.log_d) * torch.asinh(x) + self.e
        log_cosh = u.abs() + F.softplus(-2.0 * u.abs()) - math.log(2.0)
        return torch.sinh(u), (self.log_d + log_cosh - 0.5 * torch.log1p(x * x)).sum(-1)
    def inverse(self, y):
        x = torch.sinh((torch.asinh(y) - self.e) * torch.exp(-self.log_d))
        return x, -self.forward(x)[1]


class Coupling(nn.Module):
    """x_a (first n_masked coordinates) is left unchanged and conditions an elementwise transform of x_b (the rest)."""
    def __init__(self, n_masked, hidden, n_out_per_unit):
        super().__init__()
        self.n_a, self.n_b = n_masked, 3 - n_masked
        self.net = nn.Sequential(nn.Linear(self.n_a, hidden), nn.Tanh(), nn.Linear(hidden, self.n_b * n_out_per_unit))
        nn.init.zeros_(self.net[2].weight); nn.init.zeros_(self.net[2].bias)        # untrained layer = identity


class AffineCoupling(Coupling):
    def __init__(self, n_masked, hidden=32, log_scale_max=1.5):
        super().__init__(n_masked, hidden, 2)
        self.S = log_scale_max
    def _st(self, xa):
        out = self.net(xa).reshape(-1, self.n_b, 2)
        return out[..., 0], self.S * torch.tanh(out[..., 1])                         # shift, bounded log-scale
    def forward(self, x):
        xa, xb = x[:, :self.n_a], x[:, self.n_a:]
        t, ls = self._st(xa)
        return torch.cat([xa, xb * torch.exp(ls) + t], 1), ls.sum(-1)
    def inverse(self, y):
        ya, yb = y[:, :self.n_a], y[:, self.n_a:]
        t, ls = self._st(ya)
        return torch.cat([ya, (yb - t) * torch.exp(-ls)], 1), -ls.sum(-1)


def rq_spline(x, w, h, d, B, inverse=False):
    """Monotone rational-quadratic spline (Durkan et al. 2019) on [-B, B], identity outside.
       x [M]; bin widths w, heights h [M, K] (each row sums to 2B); interior knot slopes d [M, K-1] > 0 (boundary slopes = 1).
       Returns (y, log|dy/dx|), or for inverse=True (x_of_y, log|dx/dy|)."""
    M, K = w.shape
    one = torch.ones(M, 1, dtype=w.dtype, device=w.device); zero = torch.zeros_like(one)
    d_full = torch.cat([one, d, one], -1)
    kx = torch.cat([zero, torch.cumsum(w, -1)], -1) - B                       # knot positions, input axis  [M, K+1]
    ky = torch.cat([zero, torch.cumsum(h, -1)], -1) - B                       # knot positions, output axis
    inside = (x > -B) & (x < B)
    xc = torch.clamp(x, -B, B)                                                # outside points are evaluated at the edge, then masked
    idx = (xc[:, None] >= (ky if inverse else kx)[:, 1:K]).sum(-1)            # bin index 0..K-1
    pick = lambda a, i: torch.gather(a, 1, i[:, None])[:, 0]
    x_k, w_k, y_k, h_k = pick(kx, idx), pick(w, idx), pick(ky, idx), pick(h, idx)
    d_k, d_k1 = pick(d_full, idx), pick(d_full, idx + 1)
    s = h_k / w_k
    if not inverse:
        xi = (xc - x_k) / w_k; xi1 = xi * (1 - xi)
        den = s + (d_k1 + d_k - 2 * s) * xi1
        y = y_k + h_k * (s * xi ** 2 + d_k * xi1) / den
        dydx = s ** 2 * (d_k1 * xi ** 2 + 2 * s * xi1 + d_k * (1 - xi) ** 2) / den ** 2
        return torch.where(inside, y, x), torch.where(inside, torch.log(dydx), torch.zeros_like(x))
    yp = xc - y_k; t = d_k1 + d_k - 2 * s
    a = h_k * (s - d_k) + yp * t
    b = h_k * d_k - yp * t
    c = -s * yp
    xi = 2 * c / (-b - torch.sqrt(torch.clamp(b * b - 4 * a * c, min=0.0)))
    xi1 = xi * (1 - xi); den = s + t * xi1
    dydx = s ** 2 * (d_k1 * xi ** 2 + 2 * s * xi1 + d_k * (1 - xi) ** 2) / den ** 2
    return torch.where(inside, xi * w_k + x_k, x), torch.where(inside, -torch.log(dydx), torch.zeros_like(x))


class SplineCoupling(Coupling):
    def __init__(self, n_masked, K=16, hidden=32, B=6.0, min_w=1e-2, min_slope=1e-3):
        super().__init__(n_masked, hidden, 3 * K - 1)
        self.K, self.B, self.min_w, self.min_slope = K, B, min_w, min_slope
        b_slope = np.log(np.expm1(1.0 - min_slope))                                 # softplus(b) + min_slope = 1
        with torch.no_grad():
            self.net[2].bias.copy_(T(np.tile(np.r_[np.zeros(2 * K), np.full(K - 1, b_slope)], self.n_b)))
    def _params(self, xa):
        K = self.K
        wl, hl, sl = self.net(xa).reshape(-1, self.n_b, 3 * K - 1).split([K, K, K - 1], dim=-1)
        span = 2.0 * self.B - K * self.min_w
        flat = lambda t: t.reshape(-1, t.shape[-1])                                 # [N * n_b, .]
        return (flat(F.softmax(wl, -1) * span + self.min_w), flat(F.softmax(hl, -1) * span + self.min_w),
                flat(F.softplus(sl) + self.min_slope))
    def forward(self, x):
        xa, xb = x[:, :self.n_a], x[:, self.n_a:]
        yb, ld = rq_spline(xb.reshape(-1), *self._params(xa), self.B)
        return torch.cat([xa, yb.reshape(-1, self.n_b)], 1), ld.reshape(-1, self.n_b).sum(-1)
    def inverse(self, y):
        ya, yb = y[:, :self.n_a], y[:, self.n_a:]
        xb, ld = rq_spline(yb.reshape(-1), *self._params(ya), self.B, inverse=True)
        return torch.cat([ya, xb.reshape(-1, self.n_b)], 1), ld.reshape(-1, self.n_b).sum(-1)


LOG2PI = math.log(2.0 * math.pi)

class Flow(nn.Module):
    """x = layer_n(... layer_1(z)),  z ~ N(0, I)."""
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)
    def generate(self, z):
        ldj = torch.zeros(z.shape[0], dtype=z.dtype, device=z.device)
        for layer in self.layers:
            z, l = layer(z); ldj = ldj + l
        return z, ldj
    def sample(self, n):
        return self.generate(torch.randn(n, 3, dtype=DTYPE, device=DEVICE))[0]
    def log_prob(self, x):
        ld = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for layer in reversed(self.layers):
            x, l = layer.inverse(x); ld = ld + l
        return -0.5 * (x * x).sum(-1) - 1.5 * LOG2PI + ld


def build_flow(kind, mean, cov):
    """kind in {'gauss', 'realnvp', 'sas', 'nsf', 'sas_nsf'}: Gaussian-base flows that start (almost) at N(mean, cov)."""
    layers = []
    if kind == 'realnvp':
        for k in range(6):
            layers += [AffineCoupling(1 if k % 2 == 0 else 2), Permute([1, 2, 0])]
    if kind in ('nsf', 'sas_nsf'):
        for k in range(4):
            layers += [SplineCoupling(1 if k % 2 == 0 else 2), Permute([1, 2, 0])]
    if kind in ('sas', 'sas_nsf'):
        for _ in range(2 if kind == 'sas' else 1):
            layers += [Affine(np.zeros(3), np.eye(3)), SinhArcsinh()]
    layers.append(Affine(mean, cov))
    return Flow(layers).to(DEVICE)


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## Levy (alpha-stable) latent with trainable `alpha` and `beta`

Per-axis i.i.d. stable variables `z_i` with stability index `alpha_i in (1.1, 2)` and skewness `beta_i in (-1, 1)` (Chambers-Mallows-Stuck sampling from fixed
uniform / exponential noise, so it is differentiable in `alpha` and `beta`), followed by the trainable affine map. `alpha = 2` is `N(0,1)` (scaled by `1/sqrt(2)`).
The stable density has no closed form, so this model has `sample` but no `log_prob`; it is trained with sample-based losses (correlator, sliced Wasserstein).
Samples are clipped to `|z| <= 30` against `inf` / `NaN`. The sampler was checked against `scipy.stats.levy_stable`.

In [ ]:
class LevyBase(nn.Module):
    """alpha_i = alpha_min + (2 - alpha_min) sigmoid(raw_alpha_i);  beta_i = tanh(raw_beta_i);  S1 parametrisation, scale 1/sqrt(2)."""
    def __init__(self, alpha0=1.9, beta0=0.0, alpha_min=1.1, learn_alpha=True, learn_beta=True, clip=30.0):
        super().__init__()
        self.alpha_min, self.clip = alpha_min, clip
        p = (alpha0 - alpha_min) / (2.0 - alpha_min)
        self.raw_alpha = nn.Parameter(torch.full((3,), math.log(p / (1 - p))), requires_grad=learn_alpha)
        self.raw_beta = nn.Parameter(torch.full((3,), math.atanh(beta0)), requires_grad=learn_beta)
    def alpha(self):
        return self.alpha_min + (2.0 - self.alpha_min) * torch.sigmoid(self.raw_alpha)
    def beta(self):
        return torch.tanh(self.raw_beta)
    def sample(self, n):
        a, b = self.alpha(), self.beta()
        V = (torch.rand(n, 3, device=a.device) - 0.5) * math.pi * (1 - 2e-3)                 # uniform on (-pi/2, pi/2)
        W = -torch.log(torch.rand(n, 3, device=a.device).clamp_min(1e-7))                   # Exp(1)
        t = torch.tan(math.pi * a / 2)
        B = torch.atan(b * t) / a
        S = (1 + (b * t) ** 2) ** (1 / (2 * a))
        c = torch.cos(V - a * (V + B)).clamp_min(1e-12)
        x = S * torch.sin(a * (V + B)) / torch.cos(V) ** (1 / a) * (c / W) ** ((1 - a) / a)
        return (x / math.sqrt(2.0)).clamp(-self.clip, self.clip)


class LevyFlow(nn.Module):
    """r = Affine(z),  z ~ per-axis Levy-stable.  Has `sample` but no `log_prob`."""
    def __init__(self, mean, cov, alpha0=1.9, beta0=0.0, **kw):
        super().__init__()
        self.base, self.affine = LevyBase(alpha0, beta0, **kw), Affine(mean, cov)
    def sample(self, n):
        return self.affine(self.base.sample(n))[0]
    def summary(self):
        with torch.no_grad():
            print('alpha (x, y, z):', self.base.alpha().cpu().numpy().round(3), '   beta (x, y, z):', self.base.beta().cpu().numpy().round(3))
            print('shift:', self.affine.shift.cpu().numpy().round(2), '\nL =\n', self.affine.L().cpu().numpy().round(2))

## Sanity checks of the layers (autograd Jacobians on your machine)

Parameters are randomly perturbed so no layer is the identity. Checks: `inverse(forward(z)) = z`, the layer log-determinants agree with the autograd Jacobian
of the whole flow, and `log_prob` is finite.

In [ ]:
torch.manual_seed(1)
for kind in ['gauss', 'realnvp', 'sas', 'nsf', 'sas_nsf']:
    fl = build_flow(kind, init_mean, init_cov)
    with torch.no_grad():
        for p in fl.parameters():
            p.add_(0.15 * torch.randn_like(p))                       # make every layer non-trivial
    z = torch.randn(6, 3)
    x, ldj = fl.generate(z)
    # round trip through the layers' inverses
    zz, acc = x, torch.zeros(6)
    for layer in reversed(fl.layers):
        zz, l = layer.inverse(zz); acc = acc + l
    ld_auto = torch.stack([torch.slogdet(torch.autograd.functional.jacobian(lambda v: fl.generate(v[None])[0][0], z[i]))[1] for i in range(6)])
    print(f"{kind:8s} params {n_params(fl):6d}  round trip {float((zz - z).abs().max()):.1e}  log-det vs autograd {float((ldj - ld_auto).abs().max()):.1e}"
          f"  inverse log-det + forward {float((acc + ldj).abs().max()):.1e}  log_prob finite {bool(torch.isfinite(fl.log_prob(fl.sample(2000))).all())}")
lv = LevyFlow(init_mean, init_cov)
print('levy      params', n_params(lv), ' sample finite', bool(torch.isfinite(lv.sample(5000)).all()))

## Training helper (one loop for every loss)

In [ ]:
def fit(model, loss_fn, n_steps=3000, lr=2e-3, log_every=250, clip=10.0):
    """Adam + cosine learning-rate decay (to 3% of lr). `loss_fn()` returns a scalar tensor. Non-finite losses / gradients are skipped."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_steps, eta_min=0.03 * lr)
    hist = []
    for i in range(n_steps):
        loss = loss_fn()
        opt.zero_grad(set_to_none=True)
        if torch.isfinite(loss):
            loss.backward()
            for p in params:
                if p.grad is not None:
                    p.grad = torch.nan_to_num(p.grad, nan=0.0, posinf=0.0, neginf=0.0)
            torch.nn.utils.clip_grad_norm_(params, clip)
            opt.step()
        sched.step()
        hist.append(float(loss))
        if i % log_every == 0 or i == n_steps - 1:
            print(f'step {i:5d}  loss {np.nanmean(hist[-log_every:]):.5f}')
    return np.array(hist)


def correlator_loss(model, n_samples=1000):
    return lambda: compute_loss(model.sample(n_samples), c_tr, qs_tr)


def time_steps(model, n=10, n_samples=1000):
    """seconds per correlator-training step (forward + backward + Adam), for comparing with the TensorFlow notebook"""
    f = correlator_loss(model, n_samples)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=1e-4)
    f().backward(); t0 = time.time()
    for _ in range(n):
        opt.zero_grad(); f().backward(); opt.step()
    return (time.time() - t0) / n

## Gaussian source (correlator fit)

In [ ]:
gauss_flow = build_flow('gauss', init_mean, init_cov)
s = gauss_flow.sample(200000).detach().cpu().numpy()
print('untrained: mean', s.mean(0).round(2), '\ncov\n', np.cov(s.T).round(1))
hist_gauss = fit(gauss_flow, correlator_loss(gauss_flow), n_steps=1500, lr=5e-3)

## RealNVP source (correlator fit)

Six affine coupling layers with a rotation of the coordinates between layers, then the affine map; zero-initialised so the untrained model is the Gaussian guess.
Limitations: the log-scale is capped at `+-1.5` and the conditioner is a `tanh` MLP that saturates for large `|x|`, so with a Gaussian base the tails stay Gaussian-like;
and only 1000 samples per step set a noise floor on `<K>`.

In [ ]:
nvp_flow = build_flow('realnvp', init_mean, init_cov)
hist_nvp = fit(nvp_flow, correlator_loss(nvp_flow), n_steps=3000, lr=2e-3)

## Levy source (correlator fit; `alpha`, `beta` trained)

In [ ]:
levy_flow = LevyFlow(init_mean, init_cov, alpha0=1.9, beta0=0.0)            # starts close to Gaussian; alpha, beta are trained
s = levy_flow.sample(20000)
print('untrained: finite', bool(torch.isfinite(s).all()), ' median |r|', float(s.norm(dim=1).median()))
hist_levy = fit(levy_flow, correlator_loss(levy_flow), n_steps=3000, lr=3e-3)
levy_flow.summary()

## Compare the fitted sources with the real one

In [ ]:
n_eval = 50000
with torch.no_grad():
    models = {'Gaussian fit': gauss_flow.sample(n_eval).cpu().numpy(), 'RealNVP fit': nvp_flow.sample(n_eval).cpu().numpy(),
              'Levy fit': levy_flow.sample(n_eval).cpu().numpy()}
colors = {'true source (ppi-0.65)': 'k', 'Gaussian fit': 'tab:blue', 'RealNVP fit': 'tab:red', 'Levy fit': 'tab:green'}
samples = {'true source (ppi-0.65)': true_source[:n_eval], **models}

print(f"{'source':26s} {'rms(C - C_65)':>14s}   mean                    std")
for name, s in samples.items():
    c = correlation_np(s, qs_np[nonzero])
    print(f"{name:26s} {np.sqrt(np.mean((c - true_Cq[nonzero])**2)):14.5f}   {s.mean(0).round(2)}   {s.std(0).round(2)}")

fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
for i, label in enumerate(['x (out) [fm]', 'y (side) [fm]', 'z (long) [fm]']):
    bins = np.linspace(-45, 60, 100)
    for name, s in samples.items():
        ax[i].hist(s[:, i], bins=bins, density=True, histtype='step', color=colors[name], lw=1.8, label=name)
    ax[i].set_yscale('log'); ax[i].set_ylim(1e-5, None); ax[i].set_xlabel(label)
for name, s in samples.items():
    ax[3].hist(np.linalg.norm(s, axis=1), bins=np.linspace(0, 60, 100), density=True, histtype='step', color=colors[name], lw=1.8)
ax[3].set_xlabel('|r| [fm]'); ax[0].set_ylabel('density'); ax[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## Correlator fit with the new layers (`sas`, `nsf`)

`STEPS_C = 0` skips this.

In [ ]:
STEPS_C = 3000
flows_C = {}
if STEPS_C:
    for kind in ['sas', 'nsf']:
        fl = build_flow(kind, init_mean, init_cov)
        t0 = time.time(); print('---', kind, 'trained on the correlator,', n_params(fl), 'parameters')
        fit(fl, correlator_loss(fl), n_steps=STEPS_C, lr=2e-3)
        flows_C[kind] = fl; print(f'    {time.time() - t0:.0f} s')

## Direct density fit (no correlator)

The same models are trained to reproduce the **density of the true source samples** (`ppi-0.65.dat`, 500k points) directly, which removes the ill-posed correlator step
and shows how well a neural flow can represent this source at all.

* Gaussian-base models (`gauss`, `realnvp`, `sas`, `nsf`, `sas_nsf`): **maximum likelihood**, `loss = -mean log p(r)` on mini-batches of 4096; 50k points held out.
* Levy model: no closed-form density, so **sliced Wasserstein-1** between model samples and data (random 1-D projections, sorted, mean absolute difference).

In [ ]:
rng_np = np.random.default_rng(0)
perm = rng_np.permutation(len(true_source))
data_all = true_source[perm].astype(np.float32)
data_val, data_tr = data_all[:50000], data_all[50000:]
d_mean, d_cov = data_tr.mean(0), np.cov(data_tr.T)
data_t = T(data_tr)
print('train', data_tr.shape, ' val', data_val.shape, ' mean', d_mean.round(2), ' std', data_tr.std(0).round(2))

def batch(n=4096):
    return data_t[torch.randint(0, data_t.shape[0], (n,), device=DEVICE)]

def nll_loss(model):
    return lambda: -model.log_prob(batch()).mean()

def sliced_w1(a, b, n_proj=64):
    d = torch.randn(3, n_proj, dtype=a.dtype, device=a.device); d = d / d.norm(dim=0, keepdim=True)
    return (torch.sort(a @ d, dim=0).values - torch.sort(b @ d, dim=0).values).abs().mean()

def sw_loss(model, n=4096):
    return lambda: sliced_w1(model.sample(n), batch(n))

STEPS_D = 3000
flows_D = {}
for kind in ['gauss', 'realnvp', 'sas', 'nsf', 'sas_nsf']:
    fl = build_flow(kind, d_mean, d_cov)
    t0 = time.time(); print('---', kind, 'maximum likelihood on the density,', n_params(fl), 'parameters')
    fit(fl, nll_loss(fl), n_steps=STEPS_D, lr=2e-3)
    flows_D[kind] = fl; print(f'    {time.time() - t0:.0f} s')

print('--- levy (alpha, beta trained), sliced W1 on the density')
levy_D = LevyFlow(d_mean, d_cov, alpha0=1.9, beta0=0.0)
fit(levy_D, sw_loss(levy_D), n_steps=STEPS_D, lr=3e-3)
flows_D['levy'] = levy_D; levy_D.summary()

## Compare: density-trained vs correlator-trained, all against the true source

Columns: validation NLL (models with a density only), `rms(C - C_65)`, sliced Wasserstein-1 distance to the held-out true samples, and quantiles of `|r|`
(tail quantiles show whether the heavy tail is reproduced). The first row is the true source itself.

In [ ]:
n_ev = 20000
ev_true = data_val[:n_ev]
def rms_C(s):
    return float(np.sqrt(np.mean((correlation_np(s, qs_np[nonzero]) - true_Cq[nonzero]) ** 2)))
def sw1_np(a, b, n_proj=256, seed=0):
    d = np.random.default_rng(seed).standard_normal((3, n_proj)); d /= np.linalg.norm(d, axis=0)
    n = min(len(a), len(b)); return float(np.mean(np.abs(np.sort(a[:n] @ d, axis=0) - np.sort(b[:n] @ d, axis=0))))
@torch.no_grad()
def val_nll(model):
    if not hasattr(model, 'log_prob'):
        return None
    lp = model.log_prob(T(data_val)).cpu().numpy()
    ok = np.isfinite(lp); return float(-lp[ok].mean()), int((~ok).sum())
qtl = [50, 90, 99, 99.9]

entries = {}
for k, fl in flows_D.items(): entries[f'{k} [density fit]'] = fl
for k, fl in flows_C.items(): entries[f'{k} [correlator fit]'] = fl
for k, name in [('gauss_flow', 'gauss'), ('nvp_flow', 'realnvp'), ('levy_flow', 'levy')]:
    if k in globals(): entries[f'{name} [correlator fit]'] = globals()[k]

samples_ev = {'true source': ev_true}
with torch.no_grad():
    for name, fl in entries.items(): samples_ev[name] = fl.sample(n_ev).cpu().numpy()

hdr = f"{'model':26s} {'val NLL':>9s} {'rms(C-C65)':>11s} {'sliced W1':>10s}   " + '  '.join(f'|r| p{q:<5}' for q in qtl)
print(hdr); print('-' * len(hdr))
for name, s in samples_ev.items():
    nll = val_nll(entries[name]) if name in entries else None
    nll_s = f'{nll[0]:9.3f}' if nll else f"{'-':>9s}"
    if nll and nll[1]: nll_s += '*'
    print(f"{name:26s} {nll_s} {rms_C(s):11.5f} {sw1_np(s, data_val):10.4f}   " +
          '  '.join(f'{v:9.2f}' for v in np.percentile(np.linalg.norm(s, axis=1), qtl)))
print('(* = some validation points had non-finite log_prob and were skipped)')

colors = dict(zip(samples_ev, plt.cm.tab20(np.arange(len(samples_ev)))))
colors['true source'] = 'k'
fig, ax = plt.subplots(2, 4, figsize=(17, 7.5))
for row, tag in enumerate(['[density fit]', '[correlator fit]']):
    for name, s in samples_ev.items():
        if name != 'true source' and tag not in name: continue
        for i in range(3):
            ax[row, i].hist(s[:, i], bins=np.linspace(-45, 60, 100), density=True, histtype='step', color=colors[name], lw=1.6, label=name)
        ax[row, 3].hist(np.linalg.norm(s, axis=1), bins=np.linspace(0, 60, 100), density=True, histtype='step', color=colors[name], lw=1.6)
    for i, lab in enumerate(['x (out) [fm]', 'y (side) [fm]', 'z (long) [fm]', '|r| [fm]']):
        ax[row, i].set_xlabel(lab); ax[row, i].set_yscale('log'); ax[row, i].set_ylim(1e-5, None)
    ax[row, 0].set_ylabel(f'density, {tag}'); ax[row, 0].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

## Speed (compare with the TensorFlow notebook)

Seconds per correlator-training step (1000 samples x 1000 q points). To compare on your machine, time the same step in `flow-optimizer.ipynb`.

In [ ]:
for kind in ['gauss', 'realnvp', 'nsf']:
    m = build_flow(kind, init_mean, init_cov)
    print(f'{kind:8s} {time_steps(m):.3f} s / step')
print('levy     ', f'{time_steps(LevyFlow(init_mean, init_cov)):.3f} s / step')